In [ ]:
%env WORKDIR=/tmp/vault

In [ ]:
! pip install dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/tmp/vault/config.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

# Basic Configuration

In [ ]:
%%bash

vault secrets disable transit


vault secrets enable transit
# Create a new key named "kms" in the transit secrets engine
# AES-GCM is the default encryption algorithm used by the transit secrets engine. It is a symmetric encryption algorithm that provides both confidentiality and integrity for the data being encrypted. AES-GCM is widely used in modern cryptography and is considered secure for most applications.

vault write -f transit/keys/kms

# Rotate key every 1 month (30 days) to ensure that the key is regularly updated and to reduce the risk of compromise. 
# By default will will be able to use old keys for decryption, but new encryption operations will use the latest key version. 
# This helps to ensure that sensitive data is protected with the most up-to-date cryptographic algorithms and reduces the risk of data breaches.
vault write transit/keys/kms/config auto_rotate_period=30d


# Encrypt and Decrypt with CLI

In [ ]:
%%bash
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
CIPHERTEXT=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext: $CIPHERTEXT"

# Decrypt the ciphertext using the "kms" key in the transit secrets engine
DECRYPTED=$(vault write transit/decrypt/kms ciphertext=$CIPHERTEXT -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted: $DECRYPTED"   


# Encrypt and Decrypt with curl

In [ ]:
%%bash
# Same exercise as above but using curl insetad of the vault cli
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data '{"plaintext": "'"$PLAINTEXT"'"}' \
     $VAULT_ADDR/v1/transit/encrypt/kms | jq -r '.data.ciphertext'  > ciphertext.txt
CIPHERTEXT=$(cat ciphertext.txt)
echo "Ciphertext: $CIPHERTEXT"

# Decrypt the ciphertext using the "kms" key in the transit secrets engine
DECRYPTED=$(curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data '{"ciphertext": "'"$CIPHERTEXT"'"}' \
     $VAULT_ADDR/v1/transit/decrypt/kms | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted: $DECRYPTED"


# Batch operations

In [ ]:
%%bash
# Batch - Generate 1000 inputs for batch encryption

echo "Generating batch_input.json with 1000 unique card entries..."

# Start building JSON array
echo '{' > batch_input.json
echo '"batch_input": [' >> batch_input.json

# Generate 1000 entries, each with a unique card number derived from the index
for i in {1..1000}; do
    # Build a unique 16-digit card number: Visa prefix (4) + zero-padded index
    num=$(printf "4%015d" $i)
    card_number="${num:0:4} ${num:4:4} ${num:8:4} ${num:12:4}"

    # Base64 encode the plaintext
    encoded_text=$(echo -n "$card_number" | base64)

    # Add JSON entry (no trailing comma on the last entry)
    if [ $i -lt 1000 ]; then
        echo "    {\"plaintext\": \"$encoded_text\"}," >> batch_input.json
    else
        echo "    {\"plaintext\": \"$encoded_text\"}" >> batch_input.json
    fi

    # Progress indicator
    if [ $((i % 100)) -eq 0 ]; then
        echo "Generated $i entries..."
    fi
done

# Close JSON array
echo ']' >> batch_input.json
echo '}' >> batch_input.json

echo "Generated batch_input.json with 1000 unique entries"
echo "File size: $(wc -c < batch_input.json) bytes"
echo "Total lines: $(wc -l < batch_input.json) lines"

echo -e "\nFirst 10 lines of the file:"
head -10 batch_input.json

echo -e "\nLast 10 lines of the file:"
tail -10 batch_input.json


## Encrypt

In [ ]:
%%bash
# Execute batch encryption with 1000 inputs

echo "Executing batch encryption of 1000 entries..."
echo "Start time: $(date)"

# Execute the batch operation and measure time
start_time=$(date +%s%N)

curl -k -s \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data @batch_input.json \
    $VAULT_ADDR/v1/transit/encrypt/orders > batch_output.json

end_time=$(date +%s%N)
duration_ns=$((end_time - start_time))
duration_seconds=$(echo "scale=6; $duration_ns / 1000000000" | bc)

echo "Batch encryption completed!"
echo "Duration: ${duration_ns} nanoseconds"
echo "Duration: ${duration_seconds} seconds"

# Calculate operations per second (avoid division by zero)
if [ $duration_ns -gt 0 ]; then
    # Use bc for floating point arithmetic: (1000 * 1000000000) / duration_ns
    ops_per_second=$(echo "scale=2; 1000000000000 / $duration_ns" | bc)
    echo "Processing rate: ${ops_per_second} operations/second"
else
    echo "Processing rate: Unable to calculate (duration too small)"
fi

echo -e "\nOutput file size: $(wc -c < batch_output.json) bytes"

# Check if the operation was successful
if jq -e '.data.batch_results' batch_output.json > /dev/null 2>&1; then
    result_count=$(jq '.data.batch_results | length' batch_output.json)
    echo "Successfully encrypted $result_count entries"
    
    echo -e "\nFirst encrypted result:"
    jq '.data.batch_results[0]' batch_output.json
    
    echo -e "\nLast encrypted result:"
    jq '.data.batch_results[-1]' batch_output.json
else
    echo "Batch operation failed. Response:"
    jq '.' batch_output.json
fi


## Decrypt

### Adapt input batch file for decrypt operation

In [ ]:
%%bash
# Extract encrypted results and format for batch decryption

echo "Processing batch_output.json for decryption..."

# Use jq to build a properly formatted batch_decrypt_input.json directly
jq '{batch_input: [.data.batch_results[] | {ciphertext: .ciphertext}]}' \
    batch_output.json > batch_decrypt_input.json

echo "Created batch_decrypt_input.json for decryption"
echo "File size: $(wc -c < batch_decrypt_input.json) bytes"

echo -e "\nFirst 5 lines of decrypt input:"
head -5 batch_decrypt_input.json

echo -e "\nValidating JSON format:"
if jq empty batch_decrypt_input.json 2>/dev/null; then
    echo "JSON format is valid"
    entry_count=$(jq '.batch_input | length' batch_decrypt_input.json)
    echo "Total entries to decrypt: $entry_count"
else
    echo "JSON format is invalid"
fi


### Decrypt

In [ ]:
%%bash
# Execute batch decryption with 1000 entries

echo "Executing batch decryption of 1000 entries..."
echo "Start time: $(date)"

# Execute the batch decryption operation and measure time
start_time=$(date +%s%N)

curl -k -s \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data @batch_decrypt_input.json \
    $VAULT_ADDR/v1/transit/decrypt/orders > batch_decrypt_output.json

end_time=$(date +%s%N)
duration_ns=$((end_time - start_time))
duration_seconds=$(echo "scale=6; $duration_ns / 1000000000" | bc)

echo "Batch decryption completed!"
echo "Duration: ${duration_ns} nanoseconds"
echo "Duration: ${duration_seconds} seconds"

# Calculate operations per second (avoid division by zero)
if [ $duration_ns -gt 0 ]; then
    # Use bc for floating point arithmetic: (1000 * 1000000000) / duration_ns
    ops_per_second=$(echo "scale=2; 1000000000000 / $duration_ns" | bc)
    echo "Processing rate: ${ops_per_second} operations/second"
else
    echo "Processing rate: Unable to calculate (duration too small)"
fi

echo -e "\nOutput file size: $(wc -c < batch_decrypt_output.json) bytes"

# Check if the operation was successful
if jq -e '.data.batch_results' batch_decrypt_output.json > /dev/null 2>&1; then
    result_count=$(jq '.data.batch_results | length' batch_decrypt_output.json)
    echo "Successfully decrypted $result_count entries"
    
    echo -e "\nFirst decrypted result (base64 encoded):"
    jq '.data.batch_results[0]' batch_decrypt_output.json
    
    echo -e "\nFirst decrypted result (decoded):"
    jq -r '.data.batch_results[0].plaintext' batch_decrypt_output.json | base64 -d
    
    echo -e "\nLast decrypted result (decoded):"
    jq -r '.data.batch_results[-1].plaintext' batch_decrypt_output.json | base64 -d
    
    # Show a few more samples
    echo -e "\nSample of decrypted entries (first 5):"
    for i in {0..4}; do
        decrypted=$(jq -r ".data.batch_results[$i].plaintext" batch_decrypt_output.json | base64 -d)
        echo "Entry $((i+1)): $decrypted"
    done
else
    echo "Batch decryption failed. Response:"
    jq '.' batch_decrypt_output.json
fi

# Rotate key

In [ ]:
%%bash
# Rotate the key to demonstrate decryption of old ciphertexts with a new key version
echo "Rotating the 'kms' key to create a new version..."    

vault write -f transit/keys/kms/rotate 

In [ ]:
%%bash
# Encrypt a sample plaintext using the "kms" key in the transit secrets engine
PLAINTEXT=$(echo -n "Hello, Vault!" | base64)
CIPHERTEXT=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext: $CIPHERTEXT"

## Decrypt data with old and new keys

In [ ]:
%%bash
# Encrypt using the new key version to demonstrate that it can still be decrypted after rotation
PLAINTEXT=$(echo -n "Hello, Vault! (New Key Version)" | base64)
CIPHERTEXT_NEW=$(vault write transit/encrypt/kms plaintext=$PLAINTEXT -format=json | jq -r '.data.ciphertext')
echo "Ciphertext (New Key Version): $CIPHERTEXT_NEW"

# Decrypt the new ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_NEW=$(vault write transit/decrypt/kms ciphertext=$CIPHERTEXT_NEW -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (New Key Version): $DECRYPTED_NEW" 


In [ ]:
%%bash
CYPHERTEXT_OLD=$(cat ciphertext.txt)
echo "Old Ciphertext: $CYPHERTEXT_OLD"
# Decrypt the old ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_OLD=$(vault write transit/decrypt/kms ciphertext=$CYPHERTEXT_OLD -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (Old Key Version): $DECRYPTED_OLD"

# Rewrap

In [ ]:
%%bash
# Rewrap the old ciphertext to the new key version
CYPHERTEXT_OLD=$(cat ciphertext.txt)
echo "Old Ciphertext: $CYPHERTEXT_OLD"

# Rewrap the old ciphertext to the new key version using the "kms" key in the transit secrets engine
REWRAPPED_CIPHERTEXT=$(vault write transit/rewrap/kms ciphertext=$CYPHERTEXT_OLD -format=json | jq -r '.data.ciphertext')
echo "Rewrapped Ciphertext (New Key Version): $REWRAPPED_CIPHERTEXT"    

# Decrypt the rewrapped ciphertext using the "kms" key in the transit secrets engine
DECRYPTED_REWRAPPED=$(vault write transit/decrypt/kms ciphertext=$REWRAPPED_CIPHERTEXT -format=json | jq -r '.data.plaintext' | base64 --decode)
echo "Decrypted (Rewrapped to New Key Version): $DECRYPTED_REWRAPPED"

# Verifica decoding operations con datos incorrectos
Encrypt three plaintexts in one `batch_input` request, then alter two ciphertexts before a batch decrypt:
1. Wrong key version (`vault:vN:` → `vault:v99:`) so Vault cannot find that key version
2. Flip one character in the ciphertext payload so AES-GCM authentication fails
3. Leave the third ciphertext untouched as a control

Batch decrypt continues even when some items fail: failed entries return `error`, valid ones return `plaintext`.

In [ ]:
%%bash
# Encrypt 3 plaintexts in one batch_input request, tamper two ciphertexts, then batch decrypt

P1=$(echo -n "Alice 4111111111111111" | base64)
P2=$(echo -n "Bob 5500000000000004" | base64)
P3=$(echo -n "Carol 340000000000009" | base64)

jq -n --arg p1 "$P1" --arg p2 "$P2" --arg p3 "$P3" \
  '{batch_input: [{plaintext: $p1}, {plaintext: $p2}, {plaintext: $p3}]}' \
  > tamper_encrypt_input.json

echo "=== Batch encrypt input ==="
jq . tamper_encrypt_input.json

curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_encrypt_input.json \
     $VAULT_ADDR/v1/transit/encrypt/kms > tamper_encrypt_output.json

echo ""
echo "=== Original ciphertexts ==="
jq -r '.data.batch_results[] | .ciphertext' tamper_encrypt_output.json

C1=$(jq -r '.data.batch_results[0].ciphertext' tamper_encrypt_output.json)
C2=$(jq -r '.data.batch_results[1].ciphertext' tamper_encrypt_output.json)
C3=$(jq -r '.data.batch_results[2].ciphertext' tamper_encrypt_output.json)

# 1) Force a key version that does not exist (vault:vN:payload -> vault:v99:payload)
C1_TAMPERED="vault:v99:$(echo "$C1" | cut -d: -f3-)"

# 2) Flip the first character of the ciphertext payload (keeps vault:vN: prefix)
C2_PREFIX="$(echo "$C2" | cut -d: -f1-2):"
C2_PAYLOAD=$(echo "$C2" | cut -d: -f3-)
C2_FIRST=$(echo "$C2_PAYLOAD" | cut -c1)
C2_REST=$(echo "$C2_PAYLOAD" | cut -c2-)
if [ "$C2_FIRST" = "A" ]; then C2_NEW=B; else C2_NEW=A; fi
C2_TAMPERED="${C2_PREFIX}${C2_NEW}${C2_REST}"

jq -n --arg c1 "$C1_TAMPERED" --arg c2 "$C2_TAMPERED" --arg c3 "$C3" \
  '{batch_input: [{ciphertext: $c1}, {ciphertext: $c2}, {ciphertext: $c3}]}' \
  > tamper_decrypt_input.json

echo ""
echo "=== Tampered batch_input for decrypt ==="
jq . tamper_decrypt_input.json

curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_decrypt_input.json \
     $VAULT_ADDR/v1/transit/decrypt/kms > tamper_decrypt_output.json

echo ""
echo "=== Batch decrypt results ==="
jq '.data.batch_results' tamper_decrypt_output.json

echo ""
echo "=== Decoded successful entries ==="
jq -r '.data.batch_results[] | select(.plaintext != null) | .plaintext' tamper_decrypt_output.json \
  | while read -r b64; do
        echo "$b64" | base64 --decode
        echo
    done


# Verifica que es posible hacer batch decoding request con cyphertext para distintas versiones

In [ ]:
%%bash
# Verifica que es posible hacer batch decoding request con cyphertext para distintas versiones

# Encrypt 3 plaintexts in one batch_input request, tamper two ciphertexts, then batch decrypt

P1=$(echo -n "Alice 4111111111111111" | base64)
P2=$(echo -n "Bob 5500000000000004" | base64)
P3=$(echo -n "Carol 340000000000009" | base64)

jq -n --arg p1 "$P1" --arg p2 "$P2" --arg p3 "$P3" \
  '{batch_input: [{plaintext: $p1}, {plaintext: $p2}, {plaintext: $p3}]}' \
  > tamper_encrypt_input.json

curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_encrypt_input.json \
     $VAULT_ADDR/v1/transit/encrypt/kms > tamper_encrypt_output_v1.json


# Rotate keys
vault write -f transit/keys/kms/rotate 

# Repeate encrypt operation with  new key verison
curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_encrypt_input.json \
     $VAULT_ADDR/v1/transit/encrypt/kms > tamper_encrypt_output_v2.json

# Create a single batch_input request with all ciphertexts
jq -n \
  --argjson v1 "$(cat tamper_encrypt_output_v1.json)" \
  --argjson v2 "$(cat tamper_encrypt_output_v2.json)" \
  '{batch_input: [
      {ciphertext: $v1.data.batch_results[0].ciphertext},
      {ciphertext: $v2.data.batch_results[0].ciphertext}
    ]}' \
  > tamper_decrypt_input.json

echo ""
echo "=== Tampered batch_input for decrypt ==="

# Decode the batch_input request
jq . tamper_decrypt_input.json

# Decrypt the batch_input request
curl -k -s --header "X-Vault-Token: $VAULT_TOKEN" \
     --request POST \
     --data @tamper_decrypt_input.json \
     "$VAULT_ADDR/v1/transit/decrypt/kms" | jq .


# Ejemplo con Python

Misma operación de cifrado y descifrado, ahora con la librería oficial [`hvac`](https://python-hvac.org/). Transit exige que el plaintext viaje en **base64**; el ciphertext que devuelve Vault tiene el formato `vault:vN:...`, donde `N` es la versión de la clave.

## Configuramos TLS Auth Method

In [ ]:
%%bash
# Descargamos la CA de Vault (pki_int) del notebook 2_PKI.ipynb
# Estamos creando una dependencia circular con una CA del propio Vault - sólo como demostración
vault read -format=json pki_int/cert/ca_chain | jq -r .data.ca_chain > ${WORKDIR}/vault_ca.pem

In [ ]:
%%bash
# Creamos TLS Auth Method
vault auth disable cert
vault auth enable cert

# Creamos policy transit
vault policy write transit - <<EOF
path "transit/encrypt/kms" {
    capabilities = ["update"]
}
path "transit/decrypt/kms" {
    capabilities = ["update"]
}
EOF

# Creamos certificado para la autenticación
vault write auth/cert/certs/transit display_name="transit" certificate=@${WORKDIR}/vault_ca.pem ttl=1m policies="transit" \
    token_type="batch"


In [ ]:
%%bash
# Creamos certificado para la autenticación
vault write -format=json pki_int/issue/example-dot-com common_name="*.test.com" ip_sans="8.8.8.9" \
    uri_sans="otrauri.example.com,masuri.example.com" ttl="1d" > ${WORKDIR}/app_cert.json

# Extraemos el certificado y la clave privada
jq -r .data.certificate ${WORKDIR}/app_cert.json > ${WORKDIR}/vault_cert.pem
jq -r .data.private_key ${WORKDIR}/app_cert.json > ${WORKDIR}/vault_key.pem


In [ ]:
! pip install hvac

## Aplicación Python con TLS Auth Method

In [ ]:
import base64
import hvac

WORKDIR = "/tmp/vault"

# Reuses VAULT_ADDR / VAULT_CACERT loaded from dotenv / Uses TLS Auth Method
# hvac follows requests: cert is a (cert_path, key_path) tuple. There is no key= argument;
# extra kwargs are forwarded to Adapter and raise TypeError.
client = hvac.Client(
    url=VAULT_ADDR,
    cert=(f"{WORKDIR}/vault_cert.pem", f"{WORKDIR}/vault_key.pem"),
    verify=VAULT_CACERT,
)
login = client.auth.cert.login(name="transit")
auth = login["auth"] if isinstance(login, dict) and "auth" in login else login

if not client.is_authenticated():
    raise RuntimeError("hvac client is not authenticated. Check VAULT_TOKEN / VAULT_ADDR.")

token = auth.get("client_token") or client.token
token_type = auth.get("token_type", "unknown")
prefix = token.split(".", 1)[0] if token else ""
print(f"Vault token: {token}")
print(f"Token type:  {token_type}")  # batch | service
print(f"Prefix:      {prefix}.  (hvb = batch, hvs = service)")
print(f"Renewable:   {auth.get('renewable')}")  # batch tokens are not renewable
plaintext = "Hello, Vault! (Python / hvac)"
print(f"Plaintext: {plaintext}")

# Transit requires the plaintext to be base64-encoded
encrypt_response = client.secrets.transit.encrypt_data(
    name="kms",
    plaintext=base64.b64encode(plaintext.encode("utf-8")).decode("utf-8"),
)

ciphertext = encrypt_response["data"]["ciphertext"]
print(f"Ciphertext: {ciphertext}")
print(f"Key version: {encrypt_response['data']['key_version']}")

# Decrypt: Vault returns the plaintext still base64-encoded
decrypt_response = client.secrets.transit.decrypt_data(
    name="kms",
    ciphertext=ciphertext,
)
decrypted = base64.b64decode(decrypt_response["data"]["plaintext"]).decode("utf-8")
print(f"Decrypted: {decrypted}")
assert decrypted == plaintext

# Interoperability check: decrypt the ciphertext produced earlier with curl
with open("ciphertext.txt") as f:
    old_ciphertext = f.read().strip()

old_decrypt = client.secrets.transit.decrypt_data(
    name="kms",
    ciphertext=old_ciphertext,
)
old_plaintext = base64.b64decode(old_decrypt["data"]["plaintext"]).decode("utf-8")
print(f"\nOld ciphertext (from curl): {old_ciphertext}")
print(f"Decrypted (old key version): {old_plaintext}")

# Ejemplo con Java

Misma operación de cifrado y descifrado con [Spring Cloud Vault](https://docs.spring.io/spring-cloud-vault/reference/index.html). Spring Cloud Vault auto-configura el `VaultTemplate` a partir de `spring.cloud.vault.*`; el cifrado lo hace [Spring Vault](https://docs.spring.io/spring-vault/reference/vault/vault-secret-engines.html#vault.secret.transit) (`VaultTransitOperations`). A diferencia de `hvac`, **no hace falta base64 manual**: `encrypt(key, plaintext)` y `decrypt(key, ciphertext)` lo gestionan internamente.

El proyecto se genera en `$WORKDIR/java-transit` (hace falta JDK 17+ y Maven para ejecutarlo). Usa el mismo TLS Auth Method que Python (`CERT`) para que Vault emita un **batch token**. `kv.enabled=false` evita que Spring intente montar KV como `PropertySource`; aquí solo usamos Transit.

```properties
spring.application.name=vault-transit-demo
spring.main.web-application-type=none
debug=false

spring.cloud.vault.uri=${VAULT_ADDR}
spring.cloud.vault.authentication=CERT
spring.cloud.vault.ssl.cert-auth-path=cert
spring.cloud.vault.ssl.key-store=file:client.p12
spring.cloud.vault.ssl.key-store-password=changeit
spring.cloud.vault.ssl.key-store-type=PKCS12
spring.cloud.vault.kv.enabled=false
spring.config.import=optional:vault://

transit.path=transit
transit.key=kms
```

```java
@Component
class VaultTransit {
    private final VaultOperations vault;
    private final TransitProperties props;

    VaultTransit(VaultTemplate vaultTemplate, TransitProperties props) {
        this.vault = vaultTemplate;
        this.props = props;
    }

    String encrypt(String plaintext) {
        return vault.opsForTransit(props.path()).encrypt(props.key(), plaintext);
    }

    String decrypt(String ciphertext) {
        return vault.opsForTransit(props.path()).decrypt(props.key(), ciphertext);
    }
}
```

In [ ]:
from pathlib import Path
import os
import textwrap

root = Path(os.environ.get("WORKDIR", "/tmp/vault")) / "java-transit"
java_dir = root / "src/main/java/com/hashicorp/transitdemo"
res_dir = root / "src/main/resources"
java_dir.mkdir(parents=True, exist_ok=True)
res_dir.mkdir(parents=True, exist_ok=True)

truststore = root / "truststore.jks"
keystore = root / "client.p12"
vault_addr = os.environ.get("VAULT_ADDR", "https://127.0.0.1:8443")

# CERT auth requires a client keystore. Spring Cloud Vault only applies
# spring.cloud.vault.ssl.* when Apache HttpClient or OkHttp is on the classpath.
ssl_props = f"""
spring.cloud.vault.ssl.key-store=file:{keystore}
spring.cloud.vault.ssl.key-store-password=changeit
spring.cloud.vault.ssl.key-store-type=PKCS12
"""
if vault_addr.startswith("https"):
    ssl_props += f"""
spring.cloud.vault.ssl.trust-store=file:{truststore}
spring.cloud.vault.ssl.trust-store-password=changeit
spring.cloud.vault.ssl.trust-store-type=JKS
"""

(root / "pom.xml").write_text(textwrap.dedent("""\
    <?xml version=\"1.0\" encoding=\"UTF-8\"?>
    <project xmlns=\"http://maven.apache.org/POM/4.0.0\"
             xmlns:xsi=\"http://www.w3.org/2001/XMLSchema-instance\"
             xsi:schemaLocation=\"http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd\">
      <modelVersion>4.0.0</modelVersion>

      <parent>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-parent</artifactId>
        <version>3.2.5</version>
        <relativePath/>
      </parent>

      <groupId>com.hashicorp</groupId>
      <artifactId>vault-transit-demo</artifactId>
      <version>0.0.1-SNAPSHOT</version>
      <name>vault-transit-demo</name>

      <properties>
        <java.version>17</java.version>
        <spring-cloud.version>2023.0.1</spring-cloud.version>
      </properties>

      <dependencies>
        <dependency>
          <groupId>org.springframework.boot</groupId>
          <artifactId>spring-boot-starter</artifactId>
        </dependency>
        <dependency>
          <groupId>org.springframework.cloud</groupId>
          <artifactId>spring-cloud-starter-vault-config</artifactId>
        </dependency>
        <dependency>
          <groupId>org.apache.httpcomponents.client5</groupId>
          <artifactId>httpclient5</artifactId>
        </dependency>
      </dependencies>

      <dependencyManagement>
        <dependencies>
          <dependency>
            <groupId>org.springframework.cloud</groupId>
            <artifactId>spring-cloud-dependencies</artifactId>
            <version>${spring-cloud.version}</version>
            <type>pom</type>
            <scope>import</scope>
          </dependency>
        </dependencies>
      </dependencyManagement>

      <build>
        <plugins>
          <plugin>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-maven-plugin</artifactId>
          </plugin>
        </plugins>
      </build>
    </project>
    """), encoding="utf-8")

(res_dir / "application.properties").write_text(f"""\
spring.application.name=vault-transit-demo
spring.main.web-application-type=none
debug=false

spring.cloud.vault.uri=${{VAULT_ADDR}}
spring.cloud.vault.authentication=CERT
spring.cloud.vault.ssl.cert-auth-path=cert
spring.cloud.vault.kv.enabled=false
spring.cloud.vault.config.lifecycle.enabled=false
spring.config.import=optional:vault://
{ssl_props}
transit.path=transit
transit.key=kms

logging.level.root=WARN
logging.level.org.springframework=WARN
logging.level.com.hashicorp=INFO
""", encoding="utf-8")

(java_dir / "TransitDemoApplication.java").write_text(textwrap.dedent("""\
    package com.hashicorp.transitdemo;

    import java.nio.file.Files;
    import java.nio.file.Path;

    import org.springframework.boot.CommandLineRunner;
    import org.springframework.boot.SpringApplication;
    import org.springframework.boot.autoconfigure.SpringBootApplication;
    import org.springframework.boot.context.properties.ConfigurationProperties;
    import org.springframework.boot.context.properties.EnableConfigurationProperties;
    import org.springframework.stereotype.Component;
    import org.springframework.vault.authentication.SessionManager;
    import org.springframework.vault.core.VaultOperations;
    import org.springframework.vault.core.VaultTemplate;

    @SpringBootApplication
    @EnableConfigurationProperties(TransitProperties.class)
    public class TransitDemoApplication {

        public static void main(String[] args) {
            SpringApplication.run(TransitDemoApplication.class, args);
        }
    }

    @ConfigurationProperties(prefix = \"transit\")
    record TransitProperties(String path, String key) {
    }

    @Component
    class VaultTransit {
        private final VaultOperations vault;
        private final TransitProperties props;

        VaultTransit(VaultTemplate vaultTemplate, TransitProperties props) {
            this.vault = vaultTemplate;
            this.props = props;
        }

        String encrypt(String plaintext) {
            return vault.opsForTransit(props.path()).encrypt(props.key(), plaintext);
        }

        String decrypt(String ciphertext) {
            return vault.opsForTransit(props.path()).decrypt(props.key(), ciphertext);
        }
    }

    @Component
    class TransitDemo implements CommandLineRunner {
        private final VaultTransit transit;
        private final SessionManager sessionManager;
        private final VaultTemplate vault;

        TransitDemo(VaultTransit transit, SessionManager sessionManager, VaultTemplate vault) {
            this.transit = transit;
            this.sessionManager = sessionManager;
            this.vault = vault;
        }

        @Override
        public void run(String... args) throws Exception {
            printVaultToken();

            String plaintext = \"Hello, Vault! (Java / Spring Cloud Vault)\";
            System.out.println(\"Plaintext: \" + plaintext);

            String ciphertext = transit.encrypt(plaintext);
            System.out.println(\"Ciphertext: \" + ciphertext);

            String decrypted = transit.decrypt(ciphertext);
            System.out.println(\"Decrypted: \" + decrypted);
            if (!plaintext.equals(decrypted)) {
                throw new IllegalStateException(\"Decrypted text does not match plaintext\");
            }

            Path oldCipherFile = Path.of(System.getenv().getOrDefault(
                    \"CIPHERTEXT_FILE\", \"ciphertext.txt\"));
            if (Files.exists(oldCipherFile)) {
                String oldCiphertext = Files.readString(oldCipherFile).trim();
                System.out.println();
                System.out.println(\"Old ciphertext (from curl): \" + oldCiphertext);
                System.out.println(\"Decrypted (old key version): \"
                        + transit.decrypt(oldCiphertext));
            }
        }

        private void printVaultToken() {
            String token = sessionManager.getSessionToken().getToken();
            String prefix = token.contains(\".\") ? token.substring(0, token.indexOf('.')) : \"\";
            System.out.println(\"Vault token: \" + token);
            System.out.println(\"Prefix:      \" + prefix + \".  (hvb = batch, hvs = service)\");
            try {
                var lookup = vault.read(\"auth/token/lookup-self\");
                var data = lookup.getData();
                System.out.println(\"Token type:  \" + data.get(\"type\"));
                System.out.println(\"Renewable:   \" + data.get(\"renewable\"));
            } catch (Exception ex) {
                System.out.println(\"Token lookup-self failed: \" + ex.getMessage());
            }
            System.out.println();
        }
    }
    """), encoding="utf-8")

print(f"Proyecto Spring Cloud Vault escrito en {root}")

In [ ]:
%%bash
set -euo pipefail

ROOT="${WORKDIR:-/tmp/vault}/java-transit"

# Jupyter kernels do not source ~/.zshrc. Homebrew OpenJDK is keg-only, so
# /usr/bin/java is Apple's stub and "java -version" fails until JAVA_HOME is set.
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"

if [ -z "${JAVA_HOME:-}" ] || [ ! -x "${JAVA_HOME}/bin/java" ]; then
  for d in \
    /opt/homebrew/opt/openjdk@17 \
    /opt/homebrew/opt/openjdk@21 \
    /opt/homebrew/opt/openjdk \
    /usr/local/opt/openjdk@17 \
    /usr/local/opt/openjdk@21 \
    /usr/local/opt/openjdk
  do
    home="$d/libexec/openjdk.jdk/Contents/Home"
    if [ -x "$home/bin/java" ]; then
      export JAVA_HOME="$home"
      break
    fi
  done
fi

if [ -n "${JAVA_HOME:-}" ]; then
  export PATH="$JAVA_HOME/bin:$PATH"
fi

if ! command -v java >/dev/null 2>&1 || ! java -version >/dev/null 2>&1; then
  echo "JDK 17+ not found. Install a JDK and Maven to run this example."
  echo "Project generated at: $ROOT"
  exit 0
fi

if ! command -v mvn >/dev/null 2>&1; then
  echo "Maven not found. Install Maven and run:"
  echo "  mvn -f $ROOT/pom.xml spring-boot:run"
  echo "Project generated at: $ROOT"
  exit 0
fi

if [ -n "${VAULT_CACERT:-}" ] && [ -f "$VAULT_CACERT" ]; then
  rm -f "$ROOT/truststore.jks"
  keytool -importcert -noprompt -alias vault \
    -file "$VAULT_CACERT" \
    -keystore "$ROOT/truststore.jks" \
    -storepass changeit
  echo "Imported $VAULT_CACERT into $ROOT/truststore.jks"
fi

# Same client cert as the Python TLS login, packaged as PKCS12 for Spring CERT auth
CERT_DIR="${WORKDIR:-/tmp/vault}"
openssl pkcs12 -export \
  -in "$CERT_DIR/vault_cert.pem" \
  -inkey "$CERT_DIR/vault_key.pem" \
  -out "$ROOT/client.p12" \
  -name transit \
  -passout pass:changeit
echo "Created $ROOT/client.p12 for Spring Cloud Vault CERT authentication"

export CIPHERTEXT_FILE="$(pwd)/ciphertext.txt"
# Spring Boot binds env DEBUG to debug=true and dumps CONDITIONS EVALUATION REPORT.
unset DEBUG
mvn -f "$ROOT/pom.xml" -q spring-boot:run -Dspring-boot.run.arguments=--debug=false


# Long-lived apps in Kubernetes

The cert role issues a **batch token with TTL=1m**. Batch tokens cannot be renewed, so each app must **log in again** with the client certificate when the token is about to expire.

- Python (`hvac`): encrypt/decrypt every 20s; re-login when remaining TTL < 10s (and on 403)
- Java (Spring Cloud Vault): same loop via `@Scheduled`; session lifecycle + retry on Vault errors

Images are built into minikube (`transit-python:demo`, `transit-java:demo`) and run in namespace `transit-apps`. Watch logs: a new `LOGIN` / `NEW TOKEN` line after ~1 minute means rotation worked.

In [ ]:
%%bash
set -euo pipefail

REPO="$(pwd)"
WORKDIR="${WORKDIR:-/tmp/vault}"
NS=transit-apps
PROFILE="${MINIKUBE_PROFILE:-workshop-k8s}"
KCTX="${KUBE_CONTEXT:-$PROFILE}"

for f in vault.ca vault_cert.pem vault_key.pem; do
  if [ ! -f "$WORKDIR/$f" ]; then
    echo "Missing $WORKDIR/$f — run the TLS Auth cells first."
    exit 1
  fi
done

kube() { kubectl --context "$KCTX" "$@"; }

echo "Building images inside minikube profile ${PROFILE}..."
minikube image -p "$PROFILE" build -t transit-python:demo "$REPO/transit-apps/python"
minikube image -p "$PROFILE" build -t transit-java:demo "$REPO/transit-apps/java"

kube create namespace "$NS" --dry-run=client -o yaml | kube apply -f -

kube -n "$NS" create secret generic vault-tls \
  --from-file=ca.pem="$WORKDIR/vault.ca" \
  --from-file=client.pem="$WORKDIR/vault_cert.pem" \
  --from-file=client-key.pem="$WORKDIR/vault_key.pem" \
  --dry-run=client -o yaml | kube apply -f -

kube apply -f "$REPO/transit-apps/k8s.yaml"
kube -n "$NS" rollout restart deploy/transit-python deploy/transit-java || true
kube -n "$NS" rollout status deploy/transit-python --timeout=180s
kube -n "$NS" rollout status deploy/transit-java --timeout=300s

echo
kube -n "$NS" get pods -o wide
echo
echo "Follow token rotation (TTL=1m):"
echo "  kubectl --context $KCTX -n $NS logs -f deploy/transit-python"
echo "  kubectl --context $KCTX -n $NS logs -f deploy/transit-java"


In [ ]:
%%bash
# Wait past the 1m token TTL so logs should show a second LOGIN / NEW TOKEN
set -euo pipefail
NS=transit-apps

echo "Waiting 90s for batch token TTL (1m) to expire and apps to re-login..."
sleep 90

echo
echo "===== Python ====="
kubectl -n "$NS" logs deploy/transit-python --tail=40

echo
echo "===== Java ====="
kubectl -n "$NS" logs deploy/transit-java --tail=40


# Notas sobre hvac y Spring Cloud Vault

Tanto **hvac** como **Spring Cloud Vault** pueden usar un **batch token** de Vault (`hvb.…`). Transit encrypt/decrypt solo envían `X-Vault-Token`; Vault no exige un service token para esas APIs.

Un batch token **no se puede renovar** (`renewable=false`). La aplicación debe **volver a autenticarse** (certificado TLS en esta demo) para obtener un token nuevo. El TTL no está en la cadena opaca `hvb.`; viene de la respuesta de login (`auth.lease_duration`) o de `auth/token/lookup-self` (`ttl` / `expire_time`). Spring mapea el TTL del login a `LoginToken.getLeaseDuration()`.

Lo que cambia es el ciclo de vida de la sesión:

- **Python / hvac** no gestiona el TTL. Tras el login con certificado guarda `lease_duration` y, cuando el TTL restante es &lt; 10s, **se reautentica con el certificado antes de que expire**. En el camino feliz no hay 403.
- **Spring Cloud Vault** guarda el token en `LifecycleAwareSessionManager`. El refresh (`renew-self`) solo se ejecuta si el token es **renewable**, así que un batch token se usa hasta que Vault responde **403 invalid token**. Esta demo entonces llama a `destroy()` + `getSessionToken()`, que es otro login con certificado.

El mismo patrón proactivo que en Python es posible en Java: leer `LoginToken.getLeaseDuration()`, volver a hacer login `skew` segundos antes de que expire, y mantener el handler 403 como fallback. `spring.cloud.vault.session.lifecycle.refresh-before-expiry` no ayuda aquí: ese camino es **renovación** de token, no **re-login** con certificado.

# Vault Benchmark

[`vault-benchmark`](https://github.com/hashicorp/vault-benchmark) genera carga contra Transit: **encrypt 95%** / **decrypt 5%**, clave AES-256-GCM (`orders`). Decrypt usa el ciphertext que el tool siembra en el setup (un item).

Las pruebas corren como **Job** en el namespace `vault`, contra `https://vault.vault.svc.cluster.local:8200` (Service `vault` → active + performance standbys). Así el tráfico se reparte entre nodos; el port-forward a un solo pod no balancea. Los HCL llevan `disable_http2 = true` para que kube-proxy no pinne una conexión HTTP/2 a un único endpoint.

La celda de instalación **parchea** el 0.3.0 (`BatchInput` + `use_existing_mount`), cruza a `linux` con la arquitectura del nodo y construye la imagen `vault-benchmark:demo` dentro de minikube. Cada run (`run-vb-job.sh`) monta el HCL en un ConfigMap, token+CA en un Secret, imprime el informe, lo guarda en `vault-benchmark/results-<label>.txt` y extrapola a **items**.

1. **Unitario** — sin `batch_input`, **50 workers**. Una cifra/descifra por request.
2. **Batch 50 / 100 / 250 / 500 / 750** — `workers = 10`. Tamaños intermedios para ver el impacto del audit log según el tamaño de `batch_input`.
3. **Batch 1000** — `workers = 10`, mismo cardinal que `batch_input.json`.
4. **Batch ~9000** — `workers = 10`, por debajo de [`max_json_array_element_count`](https://developer.hashicorp.com/vault/docs/configuration/listener/tcp#max_json_array_element_count) (default 10 000).

Los HCL (unit + batches 50…9000; los intermedios salen de `gen-vb-hcl.py`) reutilizan el mount `transit/` y la clave `orders`: `random_mounts=false`, `cleanup=false`, `mount_name="transit"` y `use_existing_mount=true`. El setup valida sin crear ni modificar; el cleanup del tool no toca el mount reutilizado.


In [ ]:
%%bash
set -euo pipefail
WORKDIR="${WORKDIR:-/tmp/vault}"
REPO="$(pwd)"
VB_DIR="$REPO/vault-benchmark"
PROFILE="${MINIKUBE_PROFILE:-workshop-k8s}"
KCTX="${KUBE_CONTEXT:-$PROFILE}"
mkdir -p "$WORKDIR"

VB_VER=0.3.0
# Cross-compile for the minikube node (Jobs run linux, not the host Darwin binary).
NODE_ARCH=$(kubectl --context "$KCTX" get nodes -o jsonpath='{.items[0].status.nodeInfo.architecture}')
case "$NODE_ARCH" in
  amd64|x86_64) GOARCH=amd64 ;;
  arm64|aarch64) GOARCH=arm64 ;;
  *) echo "Unsupported node arch: $NODE_ARCH" >&2; exit 1 ;;
esac
echo "Building patched vault-benchmark ${VB_VER} for linux/${GOARCH}"

# Released 0.3.0 panics on encrypt.batch_input: gohcl cannot decode []interface{}.
# Also patch use_existing_mount so demos reuse transit/orders.
SRC="$WORKDIR/vault-benchmark-${VB_VER}"
rm -rf "$SRC"
curl -fsSL -o "$WORKDIR/vault-benchmark-${VB_VER}.tar.gz" \
  "https://github.com/hashicorp/vault-benchmark/archive/refs/tags/v${VB_VER}.tar.gz"
tar -xzf "$WORKDIR/vault-benchmark-${VB_VER}.tar.gz" -C "$WORKDIR"
perl -pi -e 's/BatchInput\s+\[\]interface\{\}/BatchInput []map[string]string/g' \
  "$SRC/benchmarktests/target_secret_transit.go"
python3 "$VB_DIR/patch-existing-transit.py" "$SRC"

( cd "$SRC" && CGO_ENABLED=0 GOOS=linux GOARCH="$GOARCH" go build -o "$VB_DIR/vault-benchmark" . )
chmod +x "$VB_DIR/vault-benchmark"
file "$VB_DIR/vault-benchmark"

echo "Building image vault-benchmark:demo inside minikube profile ${PROFILE}..."
minikube image -p "$PROFILE" build -t vault-benchmark:demo "$VB_DIR"
echo "Image ready. Jobs will target Service vault (all HA nodes)."


## Prometheus y matriz file/socket

Esta batería añade observabilidad a **cada ejecución**: métricas de Vault por nodo (incluidos tiempos de audit), CPU/RAM por contenedor del kubelet, IOPS/await/cola del disco, cuota/throttling y PSI del cgroup padre de Minikube. Scrape 5 s, tasas 20 s; resumen entre inicio+20 s y fin. Los JSON conservan queries y muestras, no solo promedios. `N/D` significa métrica ausente.

**Entorno compartido:** tres pods de Vault en un nodo Minikube con cuota efectiva de 2 CPU. CPU, disco y memoria también los usan generador, Prometheus y receptores; se informa de su coste. No es equivalente a tres máquinas físicas.

La preparación guarda la auditoría existente y la suspende temporalmente con autorización del operador; la última celda la restaura (una recreación cambia la clave HMAC). Los tests no borran Transit ni sus claves. No ejecutar el Clean Up general para restaurar solo esta batería.

`run-observed.sh` genera `vault-benchmark/runs/<UTC-scenario-label>/`: configuración, salida completa, recursos, tiempos, telemetría y logs de rotación. El resumen de cada test queda en su salida. Para regenerar el resumen e intentar sincronizarlo con Obsidian (si macOS permite acceder al vault): `python3 vault-benchmark/telemetry/report.py 20261028-prometheus-final`.

Socket UNIX: un receptor por PVC escucha `/vault/audit/benchmark.sock` y conserva los bytes en `benchmark-socket.json`. Mismo volumen y rotador que file; sin UDP y sin descartar registros. La confirmación de envío de socket no equivale a confirmación de persistencia fsync. Los receptores permanecen activos durante todos los escenarios.

La guía reproducible y el significado de las métricas están en `vault-benchmark/telemetry/README.md`. La suite completa se ejecuta con `python3 vault-benchmark/telemetry/suite.py` tras la preparación; reanuda casos pendientes y restaura la auditoría al terminar.

**Serie final para comparación:** `20261028-prometheus-final`. Se repite toda la matriz con inhibición temporal del reposo de macOS (`caffeinate`). La serie preliminar queda conservada como diagnóstico porque hubo reposos/saltos de reloj e incidencias de captura.

Antes de esta serie se corrigió `postrotate` a `pkill -HUP -x vault`: el patrón anterior también terminaba `vault-logrotate`, generando CrashLoopBackOff y retirando los pods del Service. Se conserva la rotación/compresión original y se miden sus costes. Solo se reiniciaron los sidecars; los procesos Vault permanecieron en ejecución. La serie `awake` previa queda como diagnóstico.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/telemetry/start.sh
python3 ./vault-benchmark/telemetry/scenarios.py prepare --allow-suspend-audit


## Without audit Logs

Línea base: sin dispositivos de audit y con el tune de Transit en valores por defecto (sin `audit_non_hmac_response_keys`). Compara throughput e `items/s` con las secciones siguientes; el coste del audit suele crecer con el tamaño de `batch_input`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py none


In [ ]:
! vault read sys/mounts/transit/tune

### 1. Operaciones unitarias (50 workers)

Sin `batch_input`: cada request cifra o descifra **un** PAN. Más workers (50) para saturar con payloads pequeños. Corre como Job `vb-none-unit` en el cluster.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-unit.hcl unit


### 2. Batch 50 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **50** PAN. Corre como Job `vb-none-batch-50`.



In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-50.hcl batch-50


### 3. Batch 100 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **100** PAN. Corre como Job `vb-none-batch-100`.



In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-100.hcl batch-100


### 4. Batch 250 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **250** PAN. Corre como Job `vb-none-batch-250`.



In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-250.hcl batch-250


### 5. Batch 500 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **500** PAN. Corre como Job `vb-none-batch-500`.



In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-500.hcl batch-500


### 6. Batch 750 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **750** PAN. Corre como Job `vb-none-batch-750`.



In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-750.hcl batch-750


### 7. Batch 1000 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **1000** PAN. Corre como Job `vb-none-batch-1000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-1000.hcl batch-1000


### 8. Batch ~9000 entradas

Misma mezcla 95/5, `workers = 10`. Por debajo del límite de 10 000 elementos JSON del listener. Corre como Job `vb-none-batch-9000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh none ./vault-benchmark/vb-transit-9000.hcl batch-9000


## With Audit Log

File audit en `/vault/audit/benchmark-audit.json` con `log_raw=false` y sin `exclude`. Transit no tiene claves non-HMAC: request y response se hashean. Misma secuencia de cargas que la línea base para medir el overhead.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py file-default


### 1. Operaciones unitarias (50 workers)

Sin `batch_input`: cada request cifra o descifra **un** PAN. Más workers (50) para saturar con payloads pequeños. Corre como Job `vb-file-default-unit` en el cluster.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-unit.hcl unit


### 2. Batch 50 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **50** PAN. Corre como Job `vb-file-default-batch-50`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-50.hcl batch-50


### 3. Batch 100 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **100** PAN. Corre como Job `vb-file-default-batch-100`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-100.hcl batch-100


### 4. Batch 250 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **250** PAN. Corre como Job `vb-file-default-batch-250`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-250.hcl batch-250


### 5. Batch 500 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **500** PAN. Corre como Job `vb-file-default-batch-500`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-500.hcl batch-500


### 6. Batch 750 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **750** PAN. Corre como Job `vb-file-default-batch-750`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-750.hcl batch-750


### 7. Batch 1000 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **1000** PAN. Corre como Job `vb-file-default-batch-1000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-1000.hcl batch-1000


### 8. Batch ~9000 entradas

Misma mezcla 95/5, `workers = 10`. Por debajo del límite de 10 000 elementos JSON del listener. Corre como Job `vb-file-default-batch-9000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-default ./vault-benchmark/vb-transit-9000.hcl batch-9000


## Audit log without request data

File audit con `exclude` de `/request/data` cuando `mount_type == transit`. El tune de Transit vuelve a hashear la respuesta (`audit_non_hmac_response_keys` vacío). El `batch_input` no entra en el audit; `ciphertext` y `batch_results` sí, y van con HMAC.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py file-no-request


### 1. Operaciones unitarias (50 workers)

Sin `batch_input`: cada request cifra o descifra **un** PAN. Más workers (50) para saturar con payloads pequeños. Corre como Job `vb-file-no-request-unit` en el cluster.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-unit.hcl unit


### 2. Batch 50 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **50** PAN. Corre como Job `vb-file-no-request-batch-50`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-50.hcl batch-50


### 3. Batch 100 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **100** PAN. Corre como Job `vb-file-no-request-batch-100`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-100.hcl batch-100


### 4. Batch 250 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **250** PAN. Corre como Job `vb-file-no-request-batch-250`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-250.hcl batch-250


### 5. Batch 500 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **500** PAN. Corre como Job `vb-file-no-request-batch-500`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-500.hcl batch-500


### 6. Batch 750 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **750** PAN. Corre como Job `vb-file-no-request-batch-750`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-750.hcl batch-750


### 7. Batch 1000 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **1000** PAN. Corre como Job `vb-file-no-request-batch-1000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-1000.hcl batch-1000


### 8. Batch ~9000 entradas

Misma mezcla 95/5, `workers = 10`. Por debajo del límite de 10 000 elementos JSON del listener. Corre como Job `vb-file-no-request-batch-9000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-no-request ./vault-benchmark/vb-transit-9000.hcl batch-9000


## Audit log without hmac in response for encrypt operations

Mismo file audit (exclude de `/request/data` en Transit, `log_raw=false`). `secrets tune` marca `ciphertext` y `batch_results` como non-HMAC en la respuesta, así el audit de encrypt no hashea esos campos.

Decrypt es unitario y se verifica que su plaintext conserve HMAC. La excepción `batch_results` se aplica al mount completo y también afectaría a una respuesta decrypt por lotes, fuera de esta matriz.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py file-combined


In [ ]:
! vault read sys/mounts/transit/tune

In [ ]:
! vault audit list -detailed

### 1. Operaciones unitarias (50 workers)

Sin `batch_input`: cada request cifra o descifra **un** PAN. Más workers (50) para saturar con payloads pequeños. Corre como Job `vb-file-combined-unit` en el cluster.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-unit.hcl unit


### 2. Batch 50 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **50** PAN. Corre como Job `vb-file-combined-batch-50`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-50.hcl batch-50


### 3. Batch 100 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **100** PAN. Corre como Job `vb-file-combined-batch-100`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-100.hcl batch-100


### 4. Batch 250 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **250** PAN. Corre como Job `vb-file-combined-batch-250`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-250.hcl batch-250


### 5. Batch 500 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **500** PAN. Corre como Job `vb-file-combined-batch-500`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-500.hcl batch-500


### 6. Batch 750 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **750** PAN. Corre como Job `vb-file-combined-batch-750`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-750.hcl batch-750


### 7. Batch 1000 entradas

Misma mezcla 95/5, `workers = 10`. Cada encrypt request cifra **1000** PAN. Corre como Job `vb-file-combined-batch-1000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-1000.hcl batch-1000


### 8. Batch ~9000 entradas

Misma mezcla 95/5, `workers = 10`. Por debajo del límite de 10 000 elementos JSON del listener. Corre como Job `vb-file-combined-batch-9000`.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh file-combined ./vault-benchmark/vb-transit-9000.hcl batch-9000


## Socket UNIX: audit completo

Misma carga y telemetría que su escenario file equivalente.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py socket-default


### 1. unit (50 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-unit.hcl unit


### 2. batch-50 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-50.hcl batch-50


### 3. batch-100 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-100.hcl batch-100


### 4. batch-250 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-250.hcl batch-250


### 5. batch-500 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-500.hcl batch-500


### 6. batch-750 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-750.hcl batch-750


### 7. batch-1000 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-1000.hcl batch-1000


### 8. batch-9000 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-default ./vault-benchmark/vb-transit-9000.hcl batch-9000


## Socket UNIX: sin request.data

Misma carga y telemetría que su escenario file equivalente.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py socket-no-request


### 1. unit (50 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-unit.hcl unit


### 2. batch-50 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-50.hcl batch-50


### 3. batch-100 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-100.hcl batch-100


### 4. batch-250 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-250.hcl batch-250


### 5. batch-500 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-500.hcl batch-500


### 6. batch-750 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-750.hcl batch-750


### 7. batch-1000 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-1000.hcl batch-1000


### 8. batch-9000 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-no-request ./vault-benchmark/vb-transit-9000.hcl batch-9000


## Socket UNIX: sin request.data + sin HMAC en respuestas encriptadas

Misma carga y telemetría que su escenario file equivalente.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py socket-combined


### 1. unit (50 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-unit.hcl unit


### 2. batch-50 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-50.hcl batch-50


### 3. batch-100 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-100.hcl batch-100


### 4. batch-250 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-250.hcl batch-250


### 5. batch-500 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-500.hcl batch-500


### 6. batch-750 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-750.hcl batch-750


### 7. batch-1000 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-1000.hcl batch-1000


### 8. batch-9000 (10 workers)


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
bash ./vault-benchmark/run-observed.sh socket-combined ./vault-benchmark/vb-transit-9000.hcl batch-9000


## Restaurar auditoría previa y resumir en Obsidian

Ejecutar también tras una interrupción. La suite automatizada restaura en su bloque finally. Prometheus y receptores permanecen para inspección; no eliminar los receptores mientras exista el dispositivo socket.


In [ ]:
%%bash
set -euo pipefail
set -a
source /tmp/vault/config.env
set +a
python3 ./vault-benchmark/telemetry/scenarios.py restore
python3 ./vault-benchmark/telemetry/verify_restoration.py 20261028-prometheus-final
python3 ./vault-benchmark/telemetry/report.py 20261028-prometheus-final


# Pruebas con LoadBalancer

Estas pruebas usan el *performance secondary* de `11_vault_pr.ipynb` y `12_pr_tasks.ipynb`. Se despliega HAProxy como balanceador **L4/TCP** con dos listeners:

- `https://127.0.0.1:18200`: primario y secundario como `backup` (failover).
- `https://127.0.0.1:18201`: primario y PR en round-robin.

La celda valida las tres casuísticas de extremo a extremo:

1. Un **service token** cifra en el primario; al forzar el failover, el PR responde `403` porque los service tokens son locales al clúster.
2. Un **batch token huérfano** cifra en el primario y descifra el mismo ciphertext en el PR después del failover. `no_parent=true` es imprescindible para usarlo entre clústeres de Performance Replication.
3. Un **batch token huérfano** realiza tres pares encrypt/decrypt por el listener round-robin; los contadores de HAProxy prueban que ambos clústeres recibieron conexiones.

Antes de desplegar el balanceador se comprueba que Performance Replication está conectada; así un PR desactualizado no se confunde con un fallo del batch token. El failover se fuerza poniendo solo el servidor primario de ese backend en mantenimiento mediante la API runtime de HAProxy. No se detiene Vault. Un `trap` lo reactiva incluso si una aserción falla. El `Service` es de tipo `LoadBalancer`; para que la prueba sea reproducible en Minikube se accede mediante port-forward local. Los tokens no se imprimen; los batch tokens duran dos minutos.

In [ ]:
%%bash
set -euo pipefail
set -a
# shellcheck disable=SC1091
source /tmp/vault/config.env
set +a
unset VAULT_NAMESPACE

KCTX="${KUBE_CONTEXT:-workshop-k8s}"
LB_NS="vault-lb"
LB_NAME="vault-transit-lb"
LB_OWNER="vault-transit-lb-demo"
LB_WORKDIR="${WORKDIR:-/tmp/vault}/lb-demo"
LB_FAILOVER="https://127.0.0.1:18200"
LB_ROUNDROBIN="https://127.0.0.1:18201"
LB_STATS="http://127.0.0.1:18404/stats;csv"
mkdir -p "$LB_WORKDIR"

command -v kubectl >/dev/null
command -v curl >/dev/null
command -v jq >/dev/null
command -v python3 >/dev/null
test -f "$VAULT_CACERT"
kubectl --context "$KCTX" get service vault-active -n vault >/dev/null
kubectl --context "$KCTX" get service vaultpr-active -n vaultpr >/dev/null

PRIMARY_POD=$(kubectl --context "$KCTX" get pod -n vault -l vault-active=true -o jsonpath='{.items[0].metadata.name}')
SECONDARY_POD=$(kubectl --context "$KCTX" get pod -n vaultpr -l vault-active=true -o jsonpath='{.items[0].metadata.name}')
if [ -z "$PRIMARY_POD" ] || [ -z "$SECONDARY_POD" ]; then
  echo 'No se encontró el pod activo de uno de los clústeres.' >&2
  exit 1
fi

# Los audit devices de notebook 13 van replicados (local=false). vaultpr debe montar
# /vault/audit (auditStorage en 11_vault_pr.ipynb); si no, el secundario suspende el PR.
ensure_pr_audit_mount() {
  if ! kubectl --context "$KCTX" get pod -n vaultpr "$SECONDARY_POD" -o json \
    | jq -e '.spec.containers[] | select(.name=="vault") | .volumeMounts[] | select(.mountPath=="/vault/audit")' >/dev/null; then
    echo "vaultpr/${SECONDARY_POD} no monta /vault/audit; los audit devices replicados romperán el PR." >&2
    echo "Añade auditStorage en overrides_pr.yaml (11_vault_pr.ipynb) y: helm upgrade -n vaultpr vaultpr hashicorp/vault -f /tmp/vault/overrides_pr.yaml" >&2
    exit 1
  fi
}

pr_connected() {
  PRIMARY_REPLICATION=$(kubectl --context "$KCTX" exec -n vault "$PRIMARY_POD" -- \
    env -u VAULT_TOKEN vault read -format=json sys/replication/performance/status)
  SECONDARY_REPLICATION=$(kubectl --context "$KCTX" exec -n vaultpr "$SECONDARY_POD" -- \
    env -u VAULT_TOKEN vault read -format=json sys/replication/performance/status)
  jq -e '.data.mode == "primary" and ([.data.secondaries[]? | select(.connection_status == "connected")] | length > 0)' \
      >/dev/null <<<"$PRIMARY_REPLICATION" && \
    jq -e '.data.mode == "secondary" and ((.data.primaries // []) | length > 0)' \
      >/dev/null <<<"$SECONDARY_REPLICATION"
}

ensure_pr_audit_mount
if ! pr_connected; then
  echo 'Performance Replication no está conectada; no se ejecutan pruebas con datos o políticas desactualizados.' >&2
  echo 'Primario:' >&2
  jq '{mode:.data.mode,state:.data.state,secondaries:.data.secondaries}' <<<"$PRIMARY_REPLICATION" >&2
  echo 'Secundario:' >&2
  jq '{mode:.data.mode,state:.data.state,primaries:.data.primaries}' <<<"$SECONDARY_REPLICATION" >&2
  echo "Diagnóstico: kubectl --context $KCTX logs -n vaultpr $SECONDARY_POD -c vault --since=30m | grep -E 'suspending replication|failed to enable replicated audit backend'" >&2
  echo "Si el log habla de /vault/audit, falta auditStorage en vaultpr." >&2
  exit 1
fi
echo "Performance Replication conectada: ${PRIMARY_POD} -> ${SECONDARY_POD}."

if kubectl --context "$KCTX" get namespace "$LB_NS" >/dev/null 2>&1; then
  owner=$(kubectl --context "$KCTX" get namespace "$LB_NS" -o jsonpath='{.metadata.labels.app\.kubernetes\.io/part-of}')
  if [ "$owner" != "$LB_OWNER" ]; then
    echo "El namespace ${LB_NS} ya existe y no pertenece a esta demo." >&2
    exit 1
  fi
else
  kubectl --context "$KCTX" create namespace "$LB_NS"
fi
kubectl --context "$KCTX" label namespace "$LB_NS" \
  "app.kubernetes.io/part-of=${LB_OWNER}" --overwrite

kubectl --context "$KCTX" apply -f - <<'YAML'
apiVersion: v1
kind: ConfigMap
metadata:
  name: vault-transit-lb
  namespace: vault-lb
  labels:
    app.kubernetes.io/part-of: vault-transit-lb-demo
data:
  haproxy.cfg: |
    global
      log stdout format raw local0
      stats socket ipv4@0.0.0.0:9999 level admin
      stats timeout 5s

    defaults
      log global
      mode tcp
      option tcplog
      timeout connect 3s
      timeout client 30s
      timeout server 30s

    frontend vault_failover
      bind *:8200
      default_backend vault_failover

    backend vault_failover
      server primary vault-active.vault.svc.cluster.local:8200 check inter 1s fall 2 rise 1
      server secondary vaultpr-active.vaultpr.svc.cluster.local:8200 check inter 1s fall 2 rise 1 backup

    frontend vault_roundrobin
      bind *:8201
      default_backend vault_roundrobin

    backend vault_roundrobin
      balance roundrobin
      server primary vault-active.vault.svc.cluster.local:8200 check inter 1s fall 2 rise 1
      server secondary vaultpr-active.vaultpr.svc.cluster.local:8200 check inter 1s fall 2 rise 1

    listen stats
      mode http
      bind *:8404
      stats enable
      stats uri /stats
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vault-transit-lb
  namespace: vault-lb
  labels:
    app: vault-transit-lb
    app.kubernetes.io/part-of: vault-transit-lb-demo
spec:
  replicas: 1
  selector:
    matchLabels:
      app: vault-transit-lb
  template:
    metadata:
      labels:
        app: vault-transit-lb
        app.kubernetes.io/part-of: vault-transit-lb-demo
    spec:
      automountServiceAccountToken: false
      securityContext:
        runAsNonRoot: true
        runAsUser: 99
        runAsGroup: 99
        seccompProfile:
          type: RuntimeDefault
      containers:
        - name: haproxy
          image: haproxy:3.0-alpine
          imagePullPolicy: IfNotPresent
          ports:
            - {name: failover, containerPort: 8200}
            - {name: roundrobin, containerPort: 8201}
            - {name: stats, containerPort: 8404}
            - {name: runtime, containerPort: 9999}
          readinessProbe:
            httpGet: {path: /stats, port: 8404}
            periodSeconds: 2
          livenessProbe:
            httpGet: {path: /stats, port: 8404}
            initialDelaySeconds: 10
            periodSeconds: 10
          resources:
            requests: {cpu: 25m, memory: 32Mi}
            limits: {cpu: 250m, memory: 128Mi}
          securityContext:
            allowPrivilegeEscalation: false
            capabilities: {drop: ["ALL"]}
          volumeMounts:
            - name: config
              mountPath: /usr/local/etc/haproxy
              readOnly: true
      volumes:
        - name: config
          configMap:
            name: vault-transit-lb
---
apiVersion: v1
kind: Service
metadata:
  name: vault-transit-lb
  namespace: vault-lb
  labels:
    app.kubernetes.io/part-of: vault-transit-lb-demo
spec:
  type: LoadBalancer
  selector:
    app: vault-transit-lb
  ports:
    - {name: failover, port: 8200, targetPort: 8200}
    - {name: roundrobin, port: 8201, targetPort: 8201}
---
apiVersion: v1
kind: Service
metadata:
  name: vault-transit-lb-admin
  namespace: vault-lb
  labels:
    app.kubernetes.io/part-of: vault-transit-lb-demo
spec:
  type: ClusterIP
  selector:
    app: vault-transit-lb
  ports:
    - {name: stats, port: 8404, targetPort: 8404}
    - {name: runtime, port: 9999, targetPort: 9999}
YAML

kubectl --context "$KCTX" rollout restart deployment/"$LB_NAME" -n "$LB_NS" >/dev/null
kubectl --context "$KCTX" rollout status deployment/"$LB_NAME" -n "$LB_NS" --timeout=180s

start_pf() {
  local name="$1" service="$2" first_port="$3" pidfile log old i
  shift 3
  pidfile="${LB_WORKDIR}/pf-${name}.pid"
  log="${LB_WORKDIR}/pf-${name}.log"
  if [ -f "$pidfile" ]; then
    old=$(cat "$pidfile")
    if kill -0 "$old" 2>/dev/null; then
      kill "$old" 2>/dev/null || true
      wait "$old" 2>/dev/null || true
    fi
  fi
  kubectl --context "$KCTX" -n "$LB_NS" port-forward --address=127.0.0.1 \
    "service/${service}" "$@" >"$log" 2>&1 &
  echo $! > "$pidfile"
  disown "$!" 2>/dev/null || true
  for i in $(seq 1 40); do
    if ! kill -0 "$(cat "$pidfile")" 2>/dev/null; then
      cat "$log" >&2
      return 1
    fi
    if python3 - "$first_port" <<'PY'
import socket, sys
try:
    with socket.create_connection(("127.0.0.1", int(sys.argv[1])), timeout=0.25):
        pass
except OSError:
    raise SystemExit(1)
PY
    then
      return 0
    fi
    sleep 0.25
  done
  echo "No se abrió el port-forward ${name}." >&2
  cat "$log" >&2
  return 1
}

start_pf data vault-transit-lb 18200 18200:8200 18201:8201
start_pf admin vault-transit-lb-admin 18404 18404:8404 19999:9999
curl -fsS "$LB_STATS" >/dev/null

lb_command() {
  python3 - "$1" <<'PY'
import socket, sys
with socket.create_connection(("127.0.0.1", 19999), timeout=3) as sock:
    sock.sendall((sys.argv[1] + "\n").encode())
    sock.shutdown(socket.SHUT_WR)
    while sock.recv(4096):
        pass
PY
}

sessions() {
  curl -fsS "$LB_STATS" | awk -F, -v be="$1" -v srv="$2" \
    '$1==be && $2==srv {print $8; found=1} END {if (!found) exit 1}'
}

status() {
  curl -fsS "$LB_STATS" | awk -F, -v be="$1" -v srv="$2" \
    '$1==be && $2==srv {print $18; found=1} END {if (!found) exit 1}'
}

wait_status() {
  local be="$1" srv="$2" expected="$3" i current
  for i in $(seq 1 30); do
    current=$(status "$be" "$srv")
    case "$current" in
      ${expected}*) return 0 ;;
    esac
    sleep 0.25
  done
  echo "Estado inesperado: ${be}/${srv}=${current}; se esperaba ${expected}." >&2
  return 1
}

restore_primary() {
  lb_command "enable server vault_failover/primary" >/dev/null 2>&1 || true
}
trap restore_primary EXIT
restore_primary
wait_status vault_failover primary UP
wait_status vault_failover secondary UP
wait_status vault_roundrobin primary UP
wait_status vault_roundrobin secondary UP

request() {
  local url="$1" token="$2" path="$3" data="$4" out="$5"
  curl -sS --cacert "$VAULT_CACERT" -o "$out" -w '%{http_code}' \
    -H "X-Vault-Token: ${token}" -H 'Content-Type: application/json' \
    --request POST --data "$data" "${url}/v1/${path}"
}

create_token() {
  local type="$1" out code token payload actual_type
  out=$(mktemp "${LB_WORKDIR}/token.XXXXXX")
  payload=$(jq -nc --arg type "$type" \
    '{policies:["transit"],ttl:"2m",type:$type,no_parent:($type == "batch")}')
  code=$(request "$LB_FAILOVER" "$VAULT_TOKEN" auth/token/create \
    "$payload" "$out")
  if [ "$code" != 200 ]; then
    echo "No se pudo crear el token ${type}; HTTP ${code}." >&2
    jq -r '.errors[]?' "$out" >&2 || true
    rm -f "$out"
    return 1
  fi
  actual_type=$(jq -r '.auth.token_type // empty' "$out")
  if [ "$actual_type" != "$type" ]; then
    echo "Vault devolvió token_type=${actual_type}; se esperaba ${type}." >&2
    rm -f "$out"
    return 1
  fi
  token=$(jq -r '.auth.client_token // empty' "$out")
  rm -f "$out"
  test -n "$token"
  printf '%s' "$token"
}

revoke_service_token() {
  local token="$1" out code
  out=$(mktemp "${LB_WORKDIR}/revoke.XXXXXX")
  code=$(request "$LB_FAILOVER" "$VAULT_TOKEN" auth/token/revoke \
    "$(jq -nc --arg token "$token" '{token:$token}')" "$out")
  rm -f "$out"
  test "$code" = 204
}

encrypt() {
  local url="$1" token="$2" text="$3" out="$4" plaintext
  plaintext=$(printf '%s' "$text" | base64 | tr -d '\n')
  request "$url" "$token" transit/encrypt/kms \
    "$(jq -nc --arg plaintext "$plaintext" '{plaintext:$plaintext}')" "$out"
}

decrypt() {
  local url="$1" token="$2" ciphertext="$3" out="$4"
  request "$url" "$token" transit/decrypt/kms \
    "$(jq -nc --arg ciphertext "$ciphertext" '{ciphertext:$ciphertext}')" "$out"
}

echo '=== 1. Primary/backup con service token ==='
SERVICE_TOKEN=$(create_token service)
P0=$(sessions vault_failover primary)
S0=$(sessions vault_failover secondary)
code=$(encrypt "$LB_FAILOVER" "$SERVICE_TOKEN" service-token "$LB_WORKDIR/service-encrypt.json")
test "$code" = 200
ciphertext=$(jq -r '.data.ciphertext // empty' "$LB_WORKDIR/service-encrypt.json")
test -n "$ciphertext"
lb_command "disable server vault_failover/primary" >/dev/null
wait_status vault_failover primary MAINT
code=$(decrypt "$LB_FAILOVER" "$SERVICE_TOKEN" "$ciphertext" "$LB_WORKDIR/service-decrypt.json")
error=$(jq -r '.errors[0] // "sin detalle"' "$LB_WORKDIR/service-decrypt.json")
test "$code" = 403
P1=$(sessions vault_failover primary)
S1=$(sessions vault_failover secondary)
echo "Primario: HTTP 200; PR tras failover: HTTP ${code} (${error})."
echo "Conexiones de la prueba: primary=$((P1-P0)), secondary=$((S1-S0))."
restore_primary
wait_status vault_failover primary UP
revoke_service_token "$SERVICE_TOKEN"
unset SERVICE_TOKEN

echo
echo '=== 2. Primary/backup con batch token ==='
BATCH_TOKEN=$(create_token batch)
P0=$(sessions vault_failover primary)
S0=$(sessions vault_failover secondary)
code=$(encrypt "$LB_FAILOVER" "$BATCH_TOKEN" batch-failover "$LB_WORKDIR/batch-encrypt.json")
test "$code" = 200
ciphertext=$(jq -r '.data.ciphertext // empty' "$LB_WORKDIR/batch-encrypt.json")
lb_command "disable server vault_failover/primary" >/dev/null
wait_status vault_failover primary MAINT
code=$(decrypt "$LB_FAILOVER" "$BATCH_TOKEN" "$ciphertext" "$LB_WORKDIR/batch-decrypt.json")
if [ "$code" != 200 ]; then
  echo "El batch token huérfano falló en el PR; HTTP ${code}." >&2
  jq -r '.errors[]?' "$LB_WORKDIR/batch-decrypt.json" >&2 || true
  exit 1
fi
decoded=$(jq -r '.data.plaintext' "$LB_WORKDIR/batch-decrypt.json" | base64 --decode)
test "$decoded" = batch-failover
P1=$(sessions vault_failover primary)
S1=$(sessions vault_failover secondary)
echo "Cifrado en primario y descifrado en PR: HTTP 200/200; texto verificado."
echo "Conexiones de la prueba: primary=$((P1-P0)), secondary=$((S1-S0))."
restore_primary
wait_status vault_failover primary UP
unset BATCH_TOKEN

echo
echo '=== 3. Round-robin con batch token ==='
BATCH_TOKEN=$(create_token batch)
P0=$(sessions vault_roundrobin primary)
S0=$(sessions vault_roundrobin secondary)
for i in 1 2 3; do
  text="round-robin-${i}"
  code=$(encrypt "$LB_ROUNDROBIN" "$BATCH_TOKEN" "$text" "$LB_WORKDIR/rr-encrypt.json")
  if [ "$code" != 200 ]; then
    echo "Falló encrypt round-robin ${i}; HTTP ${code}." >&2
    jq -r '.errors[]?' "$LB_WORKDIR/rr-encrypt.json" >&2 || true
    exit 1
  fi
  ciphertext=$(jq -r '.data.ciphertext // empty' "$LB_WORKDIR/rr-encrypt.json")
  code=$(decrypt "$LB_ROUNDROBIN" "$BATCH_TOKEN" "$ciphertext" "$LB_WORKDIR/rr-decrypt.json")
  if [ "$code" != 200 ]; then
    echo "Falló decrypt round-robin ${i}; HTTP ${code}." >&2
    jq -r '.errors[]?' "$LB_WORKDIR/rr-decrypt.json" >&2 || true
    exit 1
  fi
  decoded=$(jq -r '.data.plaintext' "$LB_WORKDIR/rr-decrypt.json" | base64 --decode)
  test "$decoded" = "$text"
done
P1=$(sessions vault_roundrobin primary)
S1=$(sessions vault_roundrobin secondary)
PD=$((P1-P0))
SD=$((S1-S0))
test $((PD+SD)) -eq 6
test "$PD" -gt 0
test "$SD" -gt 0
if [ "$PD" -gt "$SD" ]; then DIFF=$((PD-SD)); else DIFF=$((SD-PD)); fi
test "$DIFF" -le 1
echo "Seis operaciones correctas; conexiones primary=${PD}, secondary=${SD}."
echo 'Round-robin y descifrado cruzado verificados.'
unset BATCH_TOKEN

restore_primary
trap - EXIT
echo
echo "Balanceador: $(kubectl --context "$KCTX" get service "$LB_NAME" -n "$LB_NS" -o jsonpath='{.spec.type}')"
echo "Estadísticas: http://127.0.0.1:18404/stats"
echo 'Las tres pruebas han terminado correctamente.'


# Clean Up

Elimina lo creado en este notebook: namespaces `transit-apps` y `vault-lb`, port-forwards del balanceador, Jobs/ConfigMaps/Secret de vault-benchmark, imágenes en minikube (incl. `vault-benchmark:demo`), Transit, cert auth, policy y ficheros temporales. No toca los dos clústeres de Vault, la replicación ni el PKI del notebook 2.

# Audit TCP externo — receptor fuera de Minikube

Esta sección conserva los resultados de la serie independiente del 29/10. Reproducción y topología: `vault-benchmark/telemetry/EXTERNAL_TCP.md`.


In [ ]:
# Ejecutar desde Bash tras cargar /tmp/vault/config.env (no imprimir credenciales).
# VBB_STATE_DIR=/tmp/vault/vbb-tcp-external VBB_SUITE_ID=20261029-tcp-external python3 vault-benchmark/telemetry/external_suite.py


## none / batch-500 / repetición 1

Run: `20261029T090929Z-none-batch-500-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T090929Z-none-batch-500-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-500 / repetición 1

Run: `20261029T091047Z-file-default-batch-500-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T091047Z-file-default-batch-500-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-500 / repetición 1

Run: `20261029T091226Z-tcp-default-batch-500-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T091226Z-tcp-default-batch-500-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-500 / repetición 1

Run: `20261029T091454Z-file-combined-batch-500-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T091454Z-file-combined-batch-500-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-500 / repetición 1

Run: `20261029T091633Z-tcp-combined-batch-500-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T091633Z-tcp-combined-batch-500-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-500 / repetición 2

Run: `20261029T091854Z-file-default-batch-500-r2`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T091854Z-file-default-batch-500-r2')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-500 / repetición 2

Run: `20261029T092034Z-tcp-default-batch-500-r2`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T092034Z-tcp-default-batch-500-r2')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-500 / repetición 2

Run: `20261029T092301Z-file-combined-batch-500-r2`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T092301Z-file-combined-batch-500-r2')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-500 / repetición 2

Run: `20261029T092433Z-tcp-combined-batch-500-r2`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T092433Z-tcp-combined-batch-500-r2')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-500 / repetición 2

Run: `20261029T092657Z-none-batch-500-r2`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T092657Z-none-batch-500-r2')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-500 / repetición 3

Run: `20261029T092816Z-tcp-default-batch-500-r3`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T092816Z-tcp-default-batch-500-r3')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-500 / repetición 3

Run: `20261029T093050Z-file-combined-batch-500-r3`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T093050Z-file-combined-batch-500-r3')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-500 / repetición 3

Run: `20261029T093220Z-tcp-combined-batch-500-r3`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T093220Z-tcp-combined-batch-500-r3')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-500 / repetición 3

Run: `20261029T093453Z-none-batch-500-r3`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T093453Z-none-batch-500-r3')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-500 / repetición 3

Run: `20261029T093611Z-file-default-batch-500-r3`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T093611Z-file-default-batch-500-r3')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / unit / repetición 1

Run: `20261029T093752Z-none-unit-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T093752Z-none-unit-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / unit / repetición 1

Run: `20261029T093911Z-file-default-unit-r1`. Conteo audit: **millisecond_boundary**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T093911Z-file-default-unit-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / unit / repetición 1

Run: `20261029T094033Z-tcp-default-unit-r1`. Conteo audit: **millisecond_boundary**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T094033Z-tcp-default-unit-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / unit / repetición 1

Run: `20261029T094312Z-file-combined-unit-r1`. Conteo audit: **millisecond_boundary**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T094312Z-file-combined-unit-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / unit / repetición 1

Run: `20261029T094435Z-tcp-combined-unit-r1`. Conteo audit: **millisecond_boundary**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T094435Z-tcp-combined-unit-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-50 / repetición 1

Run: `20261029T094650Z-file-default-batch-50-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T094650Z-file-default-batch-50-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-50 / repetición 1

Run: `20261029T094821Z-tcp-default-batch-50-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T094821Z-tcp-default-batch-50-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-50 / repetición 1

Run: `20261029T095045Z-file-combined-batch-50-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T095045Z-file-combined-batch-50-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-50 / repetición 1

Run: `20261029T095212Z-tcp-combined-batch-50-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T095212Z-tcp-combined-batch-50-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-50 / repetición 1

Run: `20261029T095416Z-none-batch-50-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T095416Z-none-batch-50-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-100 / repetición 1

Run: `20261029T095535Z-tcp-default-batch-100-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T095535Z-tcp-default-batch-100-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-100 / repetición 1

Run: `20261029T095803Z-file-combined-batch-100-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T095803Z-file-combined-batch-100-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-100 / repetición 1

Run: `20261029T095933Z-tcp-combined-batch-100-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T095933Z-tcp-combined-batch-100-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-100 / repetición 1

Run: `20261029T100130Z-none-batch-100-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T100130Z-none-batch-100-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-100 / repetición 1

Run: `20261029T100249Z-file-default-batch-100-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T100249Z-file-default-batch-100-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-250 / repetición 1

Run: `20261029T100426Z-file-combined-batch-250-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T100426Z-file-combined-batch-250-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-250 / repetición 1

Run: `20261029T100557Z-tcp-combined-batch-250-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T100557Z-tcp-combined-batch-250-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-250 / repetición 1

Run: `20261029T100827Z-none-batch-250-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T100827Z-none-batch-250-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-250 / repetición 1

Run: `20261029T100946Z-file-default-batch-250-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T100946Z-file-default-batch-250-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-250 / repetición 1

Run: `20261029T101120Z-tcp-default-batch-250-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T101120Z-tcp-default-batch-250-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-750 / repetición 1

Run: `20261029T101333Z-tcp-combined-batch-750-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T101333Z-tcp-combined-batch-750-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-750 / repetición 1

Run: `20261029T101607Z-none-batch-750-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T101607Z-none-batch-750-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-750 / repetición 1

Run: `20261029T101727Z-file-default-batch-750-r1`. Conteo audit: **exact**.

Incidencia anterior a la carga: dos fallos de audit de sys/metrics durante la habilitación del dispositivo, 1,843 s antes del inicio. La ventana de 60 s los arrastra; la ventana de 20 s durante la medición da cero. Evidencia: audit-failure-review.json y audit-failure-operational.log.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T101727Z-file-default-batch-750-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-750 / repetición 1

Run: `20261029T101907Z-tcp-default-batch-750-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T101907Z-tcp-default-batch-750-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-750 / repetición 1

Run: `20261029T102139Z-file-combined-batch-750-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T102139Z-file-combined-batch-750-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-1000 / repetición 1

Run: `20261029T102312Z-none-batch-1000-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T102312Z-none-batch-1000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-1000 / repetición 1

Run: `20261029T102431Z-file-default-batch-1000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T102431Z-file-default-batch-1000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-1000 / repetición 1

Run: `20261029T102612Z-tcp-default-batch-1000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T102612Z-tcp-default-batch-1000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-1000 / repetición 1

Run: `20261029T102842Z-file-combined-batch-1000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T102842Z-file-combined-batch-1000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-1000 / repetición 1

Run: `20261029T103018Z-tcp-combined-batch-1000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T103018Z-tcp-combined-batch-1000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-default / batch-9000 / repetición 1

Run: `20261029T103245Z-file-default-batch-9000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T103245Z-file-default-batch-9000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-default / batch-9000 / repetición 1

Run: `20261029T103427Z-tcp-default-batch-9000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T103427Z-tcp-default-batch-9000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## file-combined / batch-9000 / repetición 1

Run: `20261029T103712Z-file-combined-batch-9000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T103712Z-file-combined-batch-9000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## tcp-combined / batch-9000 / repetición 1

Run: `20261029T103850Z-tcp-combined-batch-9000-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T103850Z-tcp-combined-batch-9000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## none / batch-9000 / repetición 1

Run: `20261029T104117Z-none-batch-9000-r1`. Conteo audit: **not_applicable**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T104117Z-none-batch-9000-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## Diagnóstico con compresión: file-combined / batch-500 / repetición 1

Run: `20261029T104248Z-file-combined-batch-500-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T104248Z-file-combined-batch-500-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## Diagnóstico con compresión: tcp-combined / batch-500 / repetición 1

Run: `20261029T104420Z-tcp-combined-batch-500-r1`. Conteo audit: **exact**.


In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, 'vault-benchmark/telemetry')
from report import summary_table
from external_report import external_table
folder = Path('vault-benchmark/runs/20261029T104420Z-tcp-combined-batch-500-r1')
telemetry = json.loads((folder / 'telemetry.json').read_text())
print((folder / 'benchmark.txt').read_text() + '\n' + summary_table({'telemetry': telemetry}).replace('Socket MiB/s / errores de escritura', 'Socket UNIX previo (inactivo) MiB/s / errores de escritura') + '\n' + external_table(telemetry), end='')


## Interrupción breve del receptor: API y audit por separado


In [ ]:
from pathlib import Path
print(Path('vault-benchmark/runs/20261029-tcp-external-resilience.json').read_text(), end='')


## Interrupción prolongada: espera de 15 s antes de cada fase


In [ ]:
from pathlib import Path
print(Path('vault-benchmark/runs/20261029-tcp-external-resilience-extended.json').read_text(), end='')


## Control positivo al recrear el dispositivo TCP


In [ ]:
from pathlib import Path
print(Path('vault-benchmark/runs/20261029-tcp-external-reconnect-verification.json').read_text(), end='')


## Verificación del receptor entregado


In [ ]:
from pathlib import Path
print(Path('vault-benchmark/runs/20261029-tcp-external-delivery-verification.json').read_text(), end='')


<!-- VBB-EXTERNAL-TCP-START -->
# Audit TCP con receptor fuera de Minikube

Serie `20261029-tcp-external`: **50/50 casos completados**. Fecha: 29/10/2026. Esta serie tiene una línea base propia; no mezcla porcentajes con las pruebas históricas.

El receptor TCP se ejecuta como contenedor hermano de Minikube en la VM de Podman: VM de 8 CPU, Minikube con cuota de 2 CPU y receptor con cuota de 2 CPU/512 MiB. El receptor tiene volumen propio, pero ambos comparten la misma VM y el disco físico. No equivale a otra máquina. Red privada `workshop-k8s`, sin puertos publicados en el Mac.

Cinco escenarios, ocho tamaños; batch 500 tiene tres repeticiones con orden alternado, los demás una. Calentamiento inicial de 30 s, duración de 60 s, 50 workers en unitario y 10 en batch, mezcla 95/5 de peticiones encrypt/decrypt; decrypt siempre unitario. Batch 9000 contiene 8.982 elementos. Combinado significa excluir request.data y evitar HMAC en ciphertext/batch_results; log_raw=false, plaintext de decrypt unitario protegido. Eximir batch_results también afectaría al plaintext de un decrypt por lotes, que no forma parte de esta matriz.

Comparación controlada: logs nuevos en directorios por caso, fuera de los patrones de rotación histórica; no se comprimen durante la carga. El receptor escribe bytes sin parsear JSON durante la carga; el conteo y la compresión se realizan después de cada ejecución, antes de la siguiente. Esta medición no incluye analítica del audit en tiempo real. La telemetría del rotador histórico permanece activa. Se conservan los logs y resultados anteriores.

Prometheus: 11 targets, scrape cada 5 s; resumen de recursos desde inicio + 20 s hasta fin, tasas con ventana de 20 s. El receptor exporta CPU/cuota/throttle, memoria, PSI, IO por cgroup, bytes, entradas y errores. La memoria del cgroup incluye caché de ficheros. TCP y su buffer no constituyen confirmación durable por evento; se espera el cierre/drenaje y fsync del receptor al terminar cada caso.

## Rendimiento de encrypt

Elementos cifrados/s = throughput de peticiones encrypt × elementos por petición. Porcentajes frente a sin audit del mismo tamaño. Para batch 500 se muestra la mediana de las tres ejecuciones.

| Prueba | Sin audit | Fichero default | TCP externo default | Fichero combinado | TCP externo combinado |
| --- | ---: | ---: | ---: | ---: | ---: |
| Unitario | 2.286 | 1.859 (-18,7 %) | 1.879 (-17,8 %) | 1.560 (-31,8 %) | 1.574 (-31,1 %) |
| Batch 50 | 156.717 | 63.494 (-59,5 %) | 60.866 (-61,2 %) | 65.927 (-57,9 %) | 62.836 (-59,9 %) |
| Batch 100 | 239.832 | 82.374 (-65,7 %) | 79.080 (-67,0 %) | 92.979 (-61,2 %) | 87.290 (-63,6 %) |
| Batch 250 | 324.682 | 91.246 (-71,9 %) | 90.187 (-72,2 %) | 124.657 (-61,6 %) | 113.368 (-65,1 %) |
| Batch 500 | 384.933 | 101.019 (-73,8 %) | 102.993 (-73,2 %) | 135.030 (-64,9 %) | 123.301 (-68,0 %) |
| Batch 750 | 392.698 | 101.196 (-74,2 %) | 103.966 (-73,5 %) | 124.501 (-68,3 %) | 129.552 (-67,0 %) |
| Batch 1000 | 385.839 | 100.753 (-73,9 %) | 110.429 (-71,4 %) | 122.345 (-68,3 %) | 126.370 (-67,2 %) |
| Batch 9000 | 497.000 | 108.300 (-78,2 %) | 116.618 (-76,5 %) | 141.467 (-71,5 %) | 154.560 (-68,9 %) |

## Latencia p95 de encrypt

Milisegundos por petición completa. Batch 500: mediana de los p95, no p95 de una distribución agregada.

| Prueba | Sin audit | Fichero default | TCP externo default | Fichero combinado | TCP externo combinado |
| --- | ---: | ---: | ---: | ---: | ---: |
| Unitario | 85,5 | 90,6 | 79,0 | 95,0 | 93,0 |
| Batch 50 | 25,8 | 40,4 | 45,7 | 38,0 | 37,9 |
| Batch 100 | 30,9 | 64,2 | 50,6 | 52,1 | 45,9 |
| Batch 250 | 52,8 | 74,1 | 86,2 | 59,2 | 78,7 |
| Batch 500 | 55,4 | 102,2 | 103,6 | 92,5 | 95,2 |
| Batch 750 | 76,0 | 159,8 | 153,5 | 128,8 | 110,5 |
| Batch 1000 | 88,3 | 200,3 | 186,0 | 190,8 | 165,7 |
| Batch 9000 | 325,8 | 1.244,5 | 1.159,7 | 1.003,4 | 889,7 |

## Repetibilidad y recursos de batch 500

Medianas de las tres ejecuciones; rango de throughput observado, sin intervalo de confianza. CPU en cores; throttle expresa periodos con cuota agotada.

| Escenario | Encrypt/s mediana [mín–máx] | p95 ms | CPU Minikube | CPU Vault | CPU receptor externo | CPU rotador | Throttle Minikube % |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Sin audit | 384.933 [378.249–421.648] | 55,4 | 1,98 | 1,58 | 0,00 | 0,00 | 100,00 |
| Fichero default | 101.019 [98.356–103.718] | 102,2 | 1,99 | 1,72 | 0,00 | 0,00 | 100,00 |
| TCP externo default | 102.993 [102.205–104.350] | 103,6 | 1,99 | 1,72 | 0,08 | 0,00 | 100,00 |
| Fichero combinado | 135.030 [126.296–135.576] | 92,5 | 1,99 | 1,72 | 0,00 | 0,00 | 100,00 |
| TCP externo combinado | 123.301 [117.883–136.142] | 95,2 | 1,99 | 1,68 | 0,04 | 0,00 | 100,00 |

### Memoria e IO de batch 500

Medianas de las medias por ejecución, salvo RAM del receptor, que resume los picos. PSI mide tiempo con tareas esperando recursos. El await corresponde al disco de la VM, compartido por todos los contenedores. La RAM del receptor incluye caché de ficheros; su uso residual en escenarios sin TCP no representa trabajo del audit de ese caso.

| Escenario | PSI CPU/IO Minikube % | PSI memoria Minikube % | RAM receptor pico MiB | PSI IO receptor % | Await VM ms |
| --- | ---: | ---: | ---: | ---: | ---: |
| Sin audit | 66,17 / 0,13 | 0,00 | 237,8 | 0,00 | 0,27 |
| Fichero default | 74,12 / 0,22 | 0,00 | 171,2 | 0,00 | 2,70 |
| TCP externo default | 73,93 / 0,22 | 0,00 | 512,0 | 13,48 | 0,81 |
| Fichero combinado | 71,81 / 0,16 | 0,00 | 141,9 | 0,00 | 1,01 |
| TCP externo combinado | 72,38 / 0,21 | 0,00 | 512,0 | 0,00 | 1,13 |

Cada prueba conserva su tabla completa de CPU, throttling, RAM, GC, IOPS, latencia de disco, latencia del audit y métricas del receptor en `10_Transit.ipynb`; las series y consultas PromQL están en su `telemetry.json`.

## Integridad y alcance

Según el benchmark: **2.109.813 peticiones**, **422.382.145 elementos cifrados** y **105.635 descifrados** en los casos completados.

36 casos auditados con conteo exacto; 4 con intervalo por precisión temporal; 10 sin auditoría. Se comparan tanto peticiones como elementos, deduplicando por request.id. Errores de petición, parseo y validación deben ser cero para dar por válida una ejecución.

Cada conexión TCP tiene un fichero separado para impedir que fragmentos de nodos distintos se intercalen. Se conservan respuestas completas y batch_results; no se usa UDP ni se descartan logs. La pérdida de conexión y la persistencia deben evaluarse aparte del rendimiento.

Incidencia de preparación conservada: al habilitar el dispositivo de fichero para batch 750, `vault-1` registró un fallo de audit de request y otro de response para `sys/metrics` (`failed to persist salt: cannot write to readonly storage`), 1,843 s antes de iniciar la carga. La métrica original con ventana de 60 s arrastra esos incrementos; la consulta de 20 s dentro de la medición da cero, y las 8.107 peticiones encrypt / 434 decrypt del benchmark tienen audit exacto. Se conservan el log operativo, ambas series y la revisión en el directorio del caso.

| Escenario unitario | Encrypt audit intervalo | Encrypt benchmark | Decrypt audit intervalo | Decrypt benchmark |
| --- | ---: | ---: | ---: | ---: |
| file-default | 111.732–111.733 | 111.732 | 5.969 | 5.969 |
| tcp-default | 112.780–112.781 | 112.780 | 5.939 | 5.939 |
| file-combined | 93.640–93.641 | 93.640 | 4.925 | 4.925 |
| tcp-combined | 94.466–94.467 | 94.466 | 4.938 | 4.938 |

Restauración de audit y tune: **verificada**.

## Diagnóstico de compresión durante la carga

Dos ejecuciones adicionales de 60 s, batch 500 combinado. En fichero se aplica logrotate cada 2 s, umbral de 64 MiB, gzip y delaycompress; en TCP se rota al completar una línea y se comprime en el receptor externo. Son políticas distintas y una sola ejecución por variante; permiten observar dónde consume CPU la compresión, no establecer una ventaja universal de un colector.

| Escenario combinado | Encrypt/s | Variación vs. sin audit | p95 ms | CPU rotador en Minikube | CPU receptor externo |
| --- | ---: | ---: | ---: | ---: | ---: |
| Fichero, sin compresión durante carga — mediana de 3 | 135.030 | -64,9 % | 92,5 | 0,00 | 0,00 |
| Fichero, con compresión — 1 ejecución | 114.523 | -70,2 % | 102,1 | 0,15 | 0,00 |
| TCP, sin compresión durante carga — mediana de 3 | 123.301 | -68,0 % | 95,2 | 0,00 | 0,04 |
| TCP, con compresión — 1 ejecución | 126.699 | -67,1 % | 95,6 | 0,00 | 0,21 |

La referencia es la mediana de batch 500 sin audit de esta serie: 384.933 elementos/s. Ambos casos adicionales tienen 100 % de éxito y conteo exacto tras leer también los ficheros comprimidos. El rotador del caso fichero alcanza 0,30 cores de pico y compite con Vault por la cuota de Minikube. En TCP, el trabajo de compresión queda fuera de esa cuota. El resultado de TCP con compresión está dentro del rango observado en sus tres ejecuciones sin compresión.

## Interrupción y recuperación del receptor

Se hicieron dos diagnósticos: una parada/arranque breve y una repetición esperando 15 s con el receptor parado y otros 15 s después de arrancarlo antes de enviar las peticiones de recuperación. Cada petición cifra dos elementos. El segundo diagnóstico usa el receptor entregado y confirma que sólo estaba habilitado el dispositivo TCP de prueba y que el contenedor estaba parado durante la fase de indisponibilidad.

| Diagnóstico | Fase | Peticiones | HTTP 200 | Respuestas localizadas en audit |
| --- | --- | ---: | ---: | ---: |
| Breve | Antes de parar | 10 | 10 | 10 |
| Breve | Receptor parado | 3 | 3 | 0 |
| Breve | Tras arrancar | 10 | 10 | 0 |
| Con espera de 15 s | Antes de parar | 10 | 10 | 10 |
| Con espera de 15 s | Receptor parado | 3 | 3 | 0 |
| Con espera de 15 s | Tras arrancar | 10 | 10 | 0 |

**La recuperación de la API no acreditó recuperación de la entrega del audit.** En esta topología y versión de Vault no se observó reconexión efectiva dentro de la ventana ensayada. Al recrear expresamente el dispositivo TCP, un control positivo volvió a registrar ambas respuestas de encrypt/decrypt, con batch_results y HMAC de plaintext correctos. No se simularon cortes de alimentación ni se midió recuperación tras esperas más largas. HashiCorp documenta la posibilidad de omitir entradas TCP tras perder la conexión aunque la petición tenga éxito: [audit socket](https://developer.hashicorp.com/vault/docs/audit/socket).

Los IDs, fases y tiempos se conservan en los artefactos `resilience`, `resilience-extended` y `reconnect-verification`. El receptor entregado incluye nombres de fichero únicos entre reinicios, con conservación verificada del contenido anterior; las imágenes medida y entregada quedan identificadas en `delivery-verification`. Esta corrección de nombres no cambia la lógica de recepción/escritura medida ni resuelve por sí sola la reconexión del emisor.

## Interpretación de los resultados

**El límite más claro en este laboratorio es la cuota compartida de CPU de Minikube.** En batch 500 se consumen 1,98–1,99 de sus 2 cores y aparece throttling en el 100 % de los periodos; esto no significa estar detenido el 100 % del tiempo. Vault consume aproximadamente 1,6–1,7 cores. La VM utiliza alrededor de 2,8–3,0 de sus 8 CPU y dispone de unos 8 GiB de RAM. No aparecen OOM ni presión de memoria en Minikube. El receptor alcanza su límite de memoria principalmente con caché de ficheros; esa cifra no equivale a heap/RSS.

El disco compartido tiene actividad y latencia, pero las medianas de batch 500 no muestran saturación global: aproximadamente 2–5 % de ocupación y 0,27–2,70 ms de await. Sí hay espera de IO en el receptor TCP default (PSI de alrededor del 13,5 %). La evidencia permite priorizar CPU en este laboratorio, sin atribuir toda la penalización a HMAC ni descartar otros límites en máquinas distintas; no se ha hecho un perfil de CPU por función.

**Sacar el receptor de Minikube no elimina el coste principal de auditoría.** En batch 500, las medianas default son 101.019 con fichero y 102.993 con TCP; en combinado, 135.030 y 123.301. Los rangos se solapan y no hay una mejora uniforme en toda la matriz. TCP combinado llega a 154.560 elementos/s en batch 9000, pero con p95 de 890 ms, frente a 126.370 y 166 ms en batch 1000. El máximo de throughput no determina por sí solo el batch adecuado.

El siguiente experimento de capacidad con más fundamento sería aumentar la cuota CPU de Minikube y separar el generador de carga, conservando payload, mezcla, audit y telemetría. La compresión fuera de la cuota de Vault sí evita competencia local por CPU. Para auditoría que deba contabilizar todas las operaciones, este receptor TCP aislado no queda validado como único destino: la entrega ante fallos necesita un diseño adicional y una nueva prueba. La batería no representa tres máquinas independientes ni demuestra capacidad productiva de una arquitectura de Performance Replication.

Estado final: auditoría original `scep-intune-validation/` y ajuste HMAC de Transit restaurados y verificados; tres pods de Vault listos, sin reinicios durante las pruebas. Prometheus y el receptor externo permanecen disponibles para inspección, con 11 targets activos y sin puertos publicados en el Mac. El receptor TCP ya no está conectado al audit de Vault.


Reproducción: `vault-benchmark/telemetry/EXTERNAL_TCP.md`. Artefactos: `vault-benchmark/runs/<run_id>/`; manifiesto `20261029-tcp-external-plan.json`. Fuentes: [audit socket](https://developer.hashicorp.com/vault/docs/audit/socket) y [audit devices](https://developer.hashicorp.com/vault/docs/audit).
<!-- VBB-EXTERNAL-TCP-END -->
